In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, load_iris
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

# =============================================================================
# 1. Bagging using scikit-learn
# =============================================================================

print("=== scikit-learn BaggingClassifier Example ===")

# Generate dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, 
                          n_redundant=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Base classifier (Decision Tree)
base_classifier = DecisionTreeClassifier(random_state=42)

# Create Bagging classifier
bagging_classifier = BaggingClassifier(
    estimator=base_classifier,      # Base classifier
    n_estimators=100,              # Number of bootstrap samples
    random_state=42,
    bootstrap=True,                # Use bootstrap (default)
    n_jobs=-1                      # Parallel processing
)

# Training
bagging_classifier.fit(X_train, y_train)

# Prediction
y_pred = bagging_classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Bagging Accuracy: {accuracy:.4f}")

# Compare with single Decision Tree
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
single_pred = single_tree.predict(X_test)
single_accuracy = accuracy_score(y_test, single_pred)

print(f"Single Decision Tree Accuracy: {single_accuracy:.4f}")
print(f"Improvement: {accuracy - single_accuracy:.4f}")

# =============================================================================
# 2. Manual Bagging Implementation
# =============================================================================

print("\n=== Manual Bagging Implementation Example ===")

class SimpleBagging:
    def __init__(self, base_estimator, n_estimators=10, random_state=None):
        self.base_estimator = base_estimator
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.estimators = []
        
    def _bootstrap_sample(self, X, y):
        """Generate bootstrap sample"""
        n_samples = X.shape[0]
        np.random.seed(self.random_state)
        
        # Select indices with replacement
        bootstrap_indices = np.random.choice(n_samples, size=n_samples, replace=True)
        
        return X[bootstrap_indices], y[bootstrap_indices]
    
    def fit(self, X, y):
        """Train Bagging ensemble"""
        self.estimators = []
        
        for i in range(self.n_estimators):
            # Generate bootstrap sample
            X_bootstrap, y_bootstrap = self._bootstrap_sample(X, y)
            
            # Copy base classifier and train
            estimator = type(self.base_estimator)(**self.base_estimator.get_params())
            estimator.fit(X_bootstrap, y_bootstrap)
            
            self.estimators.append(estimator)
            
    def predict(self, X):
        """Predict using majority voting"""
        predictions = np.array([estimator.predict(X) for estimator in self.estimators])
        
        # Majority voting
        final_predictions = []
        for i in range(X.shape[0]):
            votes = predictions[:, i]
            unique_votes, counts = np.unique(votes, return_counts=True)
            final_predictions.append(unique_votes[np.argmax(counts)])
            
        return np.array(final_predictions)

# Use custom Bagging implementation
custom_bagging = SimpleBagging(
    base_estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=50,
    random_state=42
)

# Training
custom_bagging.fit(X_train, y_train)

# Prediction
custom_pred = custom_bagging.predict(X_test)
custom_accuracy = accuracy_score(y_test, custom_pred)

print(f"Custom Bagging Accuracy: {custom_accuracy:.4f}")

# =============================================================================
# 3. Bootstrap Sampling Visualization
# =============================================================================

print("\n=== Bootstrap Sampling Visualization ===")

# Show bootstrap sampling with small dataset
original_data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print(f"Original data: {original_data}")

np.random.seed(42)
for i in range(5):
    bootstrap_sample = np.random.choice(original_data, size=len(original_data), replace=True)
    unique_elements, counts = np.unique(bootstrap_sample, return_counts=True)
    print(f"Bootstrap sample {i+1}: {bootstrap_sample}")
    print(f"  - Unique elements: {unique_elements}")
    print(f"  - Counts: {counts}")
    print(f"  - Out-of-bag elements: {set(original_data) - set(bootstrap_sample)}")
    print()

# =============================================================================
# 4. Performance Comparison (Single Model vs Bagging)
# =============================================================================

print("=== Performance Comparison ===")

# Use Iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# Run multiple times to compare stability
single_scores = []
bagging_scores = []

for random_state in range(10):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=random_state
    )
    
    # Single Decision Tree
    single_tree = DecisionTreeClassifier(random_state=random_state)
    single_tree.fit(X_train, y_train)
    single_score = single_tree.score(X_test, y_test)
    single_scores.append(single_score)
    
    # Bagging
    bagging = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=random_state),
        n_estimators=100,
        random_state=random_state
    )
    bagging.fit(X_train, y_train)
    bagging_score = bagging.score(X_test, y_test)
    bagging_scores.append(bagging_score)

print(f"Single Tree - Mean: {np.mean(single_scores):.4f}, Std: {np.std(single_scores):.4f}")
print(f"Bagging - Mean: {np.mean(bagging_scores):.4f}, Std: {np.std(bagging_scores):.4f}")

# =============================================================================
# 5. Out-of-Bag (OOB) Evaluation
# =============================================================================

print("\n=== Out-of-Bag Evaluation ===")

# Bagging with OOB score
bagging_oob = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    oob_score=True,  # Calculate OOB score
    random_state=42
)

bagging_oob.fit(X_train, y_train)

print(f"OOB Score: {bagging_oob.oob_score_:.4f}")
print(f"Test Score: {bagging_oob.score(X_test, y_test):.4f}")

# =============================================================================
# 6. Random Forest vs Manual Bagging Comparison
# =============================================================================

print("\n=== Random Forest vs Manual Bagging ===")

# Random Forest (Bagging + Feature randomness)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_score = rf.score(X_test, y_test)

# Manual Bagging (No feature randomness)
manual_bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    random_state=42
)
manual_bagging.fit(X_train, y_train)
manual_score = manual_bagging.score(X_test, y_test)

print(f"Random Forest Score: {rf_score:.4f}")
print(f"Manual Bagging Score: {manual_score:.4f}")
print("Random Forest = Bagging + Feature randomness")

=== scikit-learn BaggingClassifier Example ===
Bagging Accuracy: 0.8800
Single Decision Tree Accuracy: 0.7900
Improvement: 0.0900

=== Manual Bagging Implementation Example ===
Custom Bagging Accuracy: 0.7800

=== Bootstrap Sampling Visualization ===
Original data: [ 1  2  3  4  5  6  7  8  9 10]
Bootstrap sample 1: [ 7  4  8  5  7 10  3  7  8  5]
  - Unique elements: [ 3  4  5  7  8 10]
  - Counts: [1 1 2 3 2 1]
  - Out-of-bag elements: {np.int64(1), np.int64(2), np.int64(6), np.int64(9)}

Bootstrap sample 2: [4 8 8 3 6 5 2 8 6 2]
  - Unique elements: [2 3 4 5 6 8]
  - Counts: [2 1 1 1 2 3]
  - Out-of-bag elements: {np.int64(1), np.int64(10), np.int64(9), np.int64(7)}

Bootstrap sample 3: [ 5  1 10  6  9  1 10  3  7  4]
  - Unique elements: [ 1  3  4  5  6  7  9 10]
  - Counts: [2 1 1 1 1 1 1 2]
  - Out-of-bag elements: {np.int64(8), np.int64(2)}

Bootstrap sample 4: [9 3 5 3 7 5 9 7 2 4]
  - Unique elements: [2 3 4 5 7 9]
  - Counts: [1 2 1 2 2 2]
  - Out-of-bag elements: {np.int64(8